# Math category

- chiama i tool se c'è un'espressione già fatta. non rielabora le informazioni per creare l'espressione
- il cross encoder non prende la risposta bene in questo caso
- tool da aggiungere:
  - testare quelli aggiunti
- altre llm da provare:
  - deepseek
  - qwen 7b con quantizzazione
  - qwen for math reasoning

In [1]:
from google.colab import userdata
from huggingface_hub import login
import os
import sys
import time

In [2]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [3]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

Repository clone...
Cloning into 'NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi'...
remote: Enumerating objects: 346, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 346 (delta 32), reused 24 (delta 24), pack-reused 304 (from 2)
Receiving objects: 100% (346/346), 35.69 MiB | 13.33 MiB/s, done.
Resolving deltas: 100% (168/168), done.
/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi


In [4]:
API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Logged in as: GliEmbeddingRuspanti (role: student)


# Model

In [5]:
!pip install -q transformers accelerate bitsandbytes langchain langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer


In [7]:
from transformers import BitsAndBytesConfig
import torch
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4 è il default consigliato
    bnb_4bit_compute_dtype=torch.float16,  # fp16 ottimale su T4
    bnb_4bit_use_double_quant=True,     # risparmio memoria aggiuntivo
)

In [8]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
model_qwen = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype="auto",
            token=HF_TOKEN,
            quantization_config=bnb_config
        )
model_qwen.eval()  # disabilita dropout, riduce overhead
tokenizer_qwen = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

# Agentic

##Tools

In [9]:
!pip install -U ddgs
!pip install -q duckduckgo-search langchain-community

from langchain_community.tools import DuckDuckGoSearchRun


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_3549/738134643.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [10]:
import math
from langchain.tools import tool
from sympy import sympify, Eq, solve
from sympy import factorint
from sympy import isprime
import re


In [11]:
@tool
def empty_tool() -> str:
  """
  An empty tool that does nothing. Use in case the other tools are not useful
  """
  return ""

In [12]:
@tool("calculator")
def calculator(expression: str) -> str:
    """
    Performs arithmetic calculations. Evaluate mathematical expressions.
    """
    return str(eval(expression))

In [13]:
@tool
def solve_equation(expressions)-> str:
      """
      Solve a mathematical equation or a system of mathematical equations.
      Set the expression of each equation as a string. Do not specify the symbols.
      If there are more equations send an array of expressions
      """

      expressions = [expressions]

      equations = []
      symbols_set = set()

      variable_names = set()

      for expr in expressions:
          variable_names.update(
              re.findall(r"[a-zA-Z_]\w*", expr)
          )

      symbols_dict = {
          name: symbols(name)
          for name in variable_names
      }

      for expr in expressions:

          if "=" in expr:

              left, right = expr.split("=")

              eq = Eq(
                  eval(left, {}, symbols_dict),
                  eval(right, {}, symbols_dict)
              )

          else:

              eq = Eq(
                  eval(expr, {}, symbols_dict),
                  0
              )

          equations.append(eq)

          symbols_set.update(eq.free_symbols)

      variables = list(symbols_set)

      result = solve(equations, variables)

      return str(result)

In [14]:
search_tool = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """
    Search the web for information.
    """

    return search_tool.invoke(query)

In [15]:
@tool
def percentage_value(
    value: float,
    total: float
) -> str:
    """
    Compute percentage value/total.
    """

    if total == 0:
        return "Division by zero"

    result = (value / total) * 100

    return str(result)

In [16]:
@tool("percentage_calculator")
def percentage_calculator(expression: str) -> str:
    """
    Calculate percentage expressions.

    Examples:
    - "12% of 50"
    - "20 percent of 400"
    - "30 percentage of 20 percentage of 40"
    - "15% of 12% of 80"
    """

    try:
        expr = expression.lower()
        expr = expr.replace("percent","%")
        expr = expr.replace("percentage","%")
        expr = expr.replace( "%", "/100")
        expr = expr.replace( "of", "*")
        result = eval(expr)

        return str(result)

    except Exception as e:

        return f"Error: I could not calculate the percentage"

In [17]:
from sympy import symbols, diff, sympify

@tool
def derivative( expression: str, variable: str = "x") -> str:
    """
    Compute derivative of an expression in the variable specified as a parameter.
    """

    try:
        x = symbols(variable)
        expr = sympify(expression)
        result = diff(expr, x)
        return str(result)

    except Exception as e:

        return f"Error: {e}"

In [18]:
from sympy import integrate

@tool
def integral( expression: str, variable: str = "x") -> str:
    """
    Compute symbolic integral in the variable specified by the parameter.
    """

    try:

        x = symbols(variable)

        expr = sympify(expression)

        result = integrate(expr, x)

        return str(result)

    except Exception as e:

        return f"Error: {e}"

In [19]:
import statistics

@tool
def mean(numbers: list[float]) -> str:
    """
    Compute arithmetic mean of the numbers sent as an argument.
    """

    return str(statistics.mean(numbers))

In [20]:
from sympy import limit

@tool
def compute_limit(expression: str, variable: str, point: float) -> str:
    """
    Compute mathematical limit of the expression in the variable specified by the argument.
    The limit is computed at the point specified by the argument.
    """

    try:

        x = symbols(variable)

        expr = sympify(expression)

        result = limit(expr, x, point)

        return str(result)

    except Exception as e:

        return f"Error: {e}"

In [21]:
@tool
def quadratic_formula(a: float,b: float,c: float) -> str:
    """
    Solve quadratic equation as a*x**2 + b*x + c=0.
    """

    from sympy import symbols, solve

    x = symbols('x')

    result = solve(
        a*x**2 + b*x + c,
        x
    )

    return str(result)

In [22]:
@tool
def factor_integer(number: int) -> str:
    """
    Prime factorization.
    """

    return str(factorint(number))

In [23]:
@tool
def circle_area(radius: float) -> str:
    """
    Compute area of a circle.
    """
    return str(math.pi * radius**2)

In [24]:
@tool
def is_prime(number: int) -> str:
    """
    Check if number is prime.
    """

    return str(isprime(number))

In [25]:
tools = [calculator, solve_equation, web_search, percentage_value, percentage_calculator, derivative,
         integral, mean, compute_limit, quadratic_formula, factor_integer, circle_area, is_prime]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
calculator
Performs arithmetic calculations. Evaluate mathematical expressions.
{'expression': {'title': 'Expression', 'type': 'string'}}
--
solve_equation
Solve a mathematical equation or a system of mathematical equations.
Set the expression of each equation as a string. Do not specify the symbols.
If there are more equations send an array of expressions
{'expressions': {'title': 'Expressions'}}
--
web_search
Search the web for information.
{'query': {'title': 'Query', 'type': 'string'}}
--
percentage_value
Compute percentage value/total.
{'value': {'title': 'Value', 'type': 'number'}, 'total': {'title': 'Total', 'type': 'number'}}
--
percentage_calculator
Calculate percentage expressions.

Examples:
- "12% of 50"
- "20 percent of 400"
- "30 percentage of 20 percentage of 40"
- "15% of 12% of 80"
{'expression': {'title': 'Expression', 'type': 'string'}}
--
derivative
Compute derivative of an expression in the variable specified as a parameter.
{'expression': {'title': 'Expression'

In [26]:
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

calculator(expression: str) -> str - Performs arithmetic calculations. Evaluate mathematical expressions.
solve_equation(expressions) -> str - Solve a mathematical equation or a system of mathematical equations.
Set the expression of each equation as a string. Do not specify the symbols.
If there are more equations send an array of expressions
web_search(query: str) -> str - Search the web for information.
percentage_value(value: float, total: float) -> str - Compute percentage value/total.
percentage_calculator(expression: str) -> str - Calculate percentage expressions.

Examples:
- "12% of 50"
- "20 percent of 400"
- "30 percentage of 20 percentage of 40"
- "15% of 12% of 80"
derivative(expression: str, variable: str = 'x') -> str - Compute derivative of an expression in the variable specified as a parameter.
integral(expression: str, variable: str = 'x') -> str - Compute symbolic integral in the variable specified by the parameter.
mean(numbers: list[float]) -> str - Compute arithme

## llm creation

In [27]:
!pip install langchain_huggingface

In [28]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer,pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

In [29]:
pipe_qwen_router = pipeline(
    "text-generation",
    model=model_qwen,
    tokenizer=tokenizer_qwen,
    max_new_tokens=100,          # sufficiente per rispondere a MCA
    do_sample=False,            # greedy: più veloce e deterministico
    temperature=None,           # obbligatorio quando do_sample=False
    top_p=None,                 # obbligatorio quando do_sample=False
    return_full_text=False,     # ritorna solo la parte generata, non il prompt
    pad_token_id=tokenizer_qwen.eos_token_id,  # evita warning su T4
)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'do_sample', 'temperature', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [30]:
pipe_qwen_answer = pipeline(
    "text-generation",
    model=model_qwen,
    tokenizer=tokenizer_qwen,
    max_new_tokens=200,          # sufficiente per rispondere a MCA
    do_sample=False,            # greedy: più veloce e deterministico
    temperature=None,           # obbligatorio quando do_sample=False
    top_p=None,                 # obbligatorio quando do_sample=False
    return_full_text=False,     # ritorna solo la parte generata, non il prompt
    pad_token_id=tokenizer_qwen.eos_token_id,  # evita warning su T4
)

In [31]:
llm_router = HuggingFacePipeline(
    pipeline=pipe_qwen_router,
    pipeline_kwargs={
        "temperature": 0.2
    }
)

In [32]:
llm_answer = HuggingFacePipeline(
    pipeline=pipe_qwen_answer,
    pipeline_kwargs={
        "temperature": 0.2
    }
)

In [33]:
chat_qwen_router = ChatHuggingFace(llm=llm_router)
chat_qwen_answer = ChatHuggingFace(llm=llm_answer)

# Model

In [34]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


modelSentence = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def pick_by_all_MiniLM(question: str, options: dict):
    labels = list(options.keys())
    texts = [options[label] for label in labels]

    # --- embeddings ---
    question_embedding = modelSentence.encode([question])
    options_embeddings = modelSentence.encode(texts)

    # --- cosine similarities ---
    scores = cosine_similarity(
        question_embedding,
        options_embeddings
    )[0]

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = (
        float(best_score - np.mean(sorted_scores[1:]))
        if len(labels) > 1 else 0.0
    )

    # --- NORMALIZED MARGIN ---
    score_range = np.max(scores) - np.min(scores) + 1e-8
    normalized_margin = (best_score - second_score) / score_range

    # --- probabilistic closeness between top2 ---
    relative_second_closeness = (
        np.exp(second_score) /
        (np.exp(best_score) + np.exp(second_score))
    )

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i])
        for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i])
        for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,
        "best_probability": best_prob,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [35]:
import numpy as np
from typing import Callable


In [36]:
class Model:
    """Base class. Subclasses implement generate().
       answer_fn decide how to get the final option."""
    def __init__(self, name: str, answer_fn: Callable):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        raise NotImplementedError

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        raw_output = self.generate(question, options, system_prompt)
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"MODEL ANSWER ----->{raw_output}")
        return summary_answer, answer

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"

In [37]:
from typing import Callable, Any, Dict, Optional, TypedDict
import json

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableConfig
)
from langchain_core.tools import render_text_description
from langchain_core.messages import AIMessage

class AgenticModel(Model):

    def __init__(
        self,
        name: str,
        router_llm,
        final_llm,
        tools,
        answer_fn: Callable
    ):

        super().__init__(name, answer_fn)

        self.router_llm = router_llm
        self.final_llm = final_llm
        self.tools = tools

        # =========================
        # TOOL DESCRIPTIONS
        # =========================

        self.rendered_tools = render_text_description(
            self.tools
        )

        # =========================
        # ROUTER SYSTEM PROMPT
        # =========================
        self.system_prompt_router = f"""\
You are a correct assistant that has access ONLY to the following tools:
{self.rendered_tools}

Your task is to select the BEST tool among the ones above.
Return your response ONLY as a JSON blob with 'name' and 'arguments' keys.
Do NOT write any introduction, explanation, or reasoning. Start your response directly with '{{'.

The 'name' should be the name of the tool to use that is part of the ones above.
The `arguments` should be a dictionary, with keys corresponding to the argument names and the values corresponding to the values requested by the user.
If there is no useful tool among the ones above, redirect to the tool with name web_search and the query string as an argument.
"""

        # =========================
        # ROUTER PROMPT
        # =========================

        self.prompt_router = ChatPromptTemplate.from_messages(
            [
                ("system", self.system_prompt_router),
                ("human", "{{input}} and the possible answers are {{options}}")
            ],
            template_format="jinja2"
        )

        # =========================
        # FINAL PROMPT
        # =========================
        self.final_prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an helpful and correct assistant answering to a MCA question."),
            ("human", """Question:
{input}

Possible answers:
{options}

Tool used:
{tool_call}

Tool output:
{output}

Use the tool output to generate the answer.
Your answer should be: I used the tool 'tool_name' and the correct answer is option 'number_of_the_correct_option': 'text_of_the_correct_option'.
Do NOT answer with anything else""")
        ])

        # =========================
        # PARSER
        # =========================

        self.custom_parser = RunnableLambda(
            self.extract_json_from_message
        )

        # =========================
        # ROUTER CHAIN
        # =========================
        self.router_chain = (
            {
                "input": RunnableLambda(
                    lambda x: x["input"]
                ),
                "options": RunnableLambda(
                    lambda x: x["options"]
                ),
                "tool_call":
                    self.prompt_router
                    | self.router_llm
                    | self.custom_parser
            }
        )


        # =========================
        # FULL CHAIN
        # =========================

        self.chain = (
            self.router_chain
            | RunnablePassthrough.assign(
                output=self.invoke_tool
            )
            | self.final_prompt
            | self.final_llm
        )

    # ==================================
    # JSON PARSER
    # ==================================
    def extract_json_from_message(self, message: Any):
        try:
            # Estrae la stringa di testo dall'oggetto AIMessage
            text = message.content if hasattr(message, "content") else str(message)

            decoder = json.JSONDecoder()
            # Non serve più fare lo split su "Assistant:"
            start = text.find("{")
            if start == -1:
                raise ValueError("Nessun JSON trovato nella risposta del modello")

            obj, _ = decoder.raw_decode(text[start:])
            print(f"-----JSON OBJECT: {obj}-----")
            return obj

        except Exception as e:
            print(f"-----PARSING ERROR: {e}-----")
            return {
                "name": "empty_tool",
                "arguments": {}
            }

    # ==================================
    # TOOL INVOCATION
    # ==================================
    def invoke_tool(
        self,
        x: Dict[str, Any],
        config: Optional[RunnableConfig] = None
    ):
        """
        A function that we can use the perform a tool invocation safely.

        Args:
            tool_call_request: a dict that contains the keys name and arguments.
                The name must match the name of a tool that exists.
                The arguments are the arguments to that tool.
            config: This is configuration information that LangChain uses that contains
                things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

        Returns:
            output from the requested tool

        Falls back to empty_tool if:
            tool_call is missing
            name is missing
            tool does not exist
            arguments are malformed
        """
        tool_name_to_tool = {
            tool.name: tool
            for tool in self.tools
        }

        try:
            print("---TOOL INVOCATION---")
            print(f"X object: {x}")
            tool_call_request = x.get(
                "tool_call",
                {}
            )

            name = tool_call_request.get(
                "name",
                "empty_tool"
            )

            if name not in tool_name_to_tool:
                print(f"-----TOOL NOT FOUND: {name}-----")
                return empty_tool.invoke({})

            requested_tool = tool_name_to_tool[
                name
            ]

            arguments = tool_call_request.get(
                "arguments",
                {}
            )

            if not isinstance(arguments, dict):

                return empty_tool.invoke({})

            return requested_tool.invoke(
                arguments,
                config=config
            )

        except Exception:

            return empty_tool.invoke({})

    # ==================================
    # GENERATE
    # ==================================

    def generate(
        self,
        question: str,
        options: dict,
        system_prompt: str = ""
    ) -> str:

        response = self.chain.invoke(
            {
                "input": question,
                "options": options
            }
        )

        print("========DEBUGGING ANSWER GENERATION=========")
        print(f"Question: {question}")
        print(f"Options: {options}")
        print(f"Response: {response}")
        print("============================================")



        # HuggingFacePipeline può restituire stringa
        # oppure AIMessage

        text = response.content if hasattr(response, "content") else str(response)

        if "<|im_start|>assistant" in text:
            text = text.split("<|im_start|>assistant")[-1]
        if "<|im_end|>" in text:
            text = text.split("<|im_end|>")[0]

        return text.strip()

# Game

In [38]:
def play_game(game, model_name, verbose=False):
    agenticModel = AgenticModel(model_name, chat_qwen_router, chat_qwen_answer, tools, pick_by_all_MiniLM)
    log = []

    while game.in_progress:
        question = game.current_question
        if not question:
            print("No question available. Game may have ended.")
            break

        print(f"\n--- Level {game.current_level} ---")
        print(f"Q: {question.text}")
        for opt in question.options:
            print(f"  [{opt.id}] {opt.text}")

        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        options = {f"{opt.id}": opt.text for opt in question.options}

        t0 = time.time()
        answer_summary, answer_input = agenticModel.answer(question.text, options)
        inference_time = time.time() - t0

        print(f"Agentic answer: {answer_input}")
        answer_id = int(answer_input)
        choosen_answer = question.options[answer_id]

        result = game.answer(answer_id)

        if result.correct:
            print(" CORRECT!")
            if result.game_over:
                print(f"\n CONGRATULATIONS! You completed the game!")
                print(f" Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f" Earned so far: ${result.earned_amount:,.2f}")
        elif result.timed_out:
            print("TIMED OUT!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        elif not result.correct:
            print(" WRONG ANSWER!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        # save outcome in the log(useful for graphs)
        entry = {
            'level'          : game.current_level,
            'question'       : question.text,
            'options'        : question.options,
            'chosen_option'  : choosen_answer.text,
            'correct'        : result.correct,
            'timed_out'      : result.timed_out,
            'inference_time' : round(inference_time, 2),
            #'answer_summary' : answer_summary,
        }
        log.append(entry)

    summary = {
        'model'          : model_name,
        'final_level'    : game.current_level,
        'earned_amount'  : game.earned_amount,
        'num_questions'  : len(log),
        'num_correct'    : sum(1 for e in log if e['correct']),
        'num_timed_out'  : sum(1 for e in log if e['timed_out']),
        'avg_inference_s': round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'            : log,
    }

    print(f"\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

    return summary

In [42]:
model_name="agenticModel"
print(f"\n########## MODEL: {model_name} ##########")

model_results = []
for comp_id in [3]:
    print(f"\n--- Competition {comp_id} ---")

    game = client.game.start(competition_id=comp_id)

    summary = play_game(game, model_name)

    model_results.append(summary)


########## MODEL: agenticModel ##########

--- Competition 3 ---


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Level 1 ---
Q: Five thousand dollars compounded annually at an $x\%$ interest rate takes six years to double. At the same interest rate, how many years will it take $\$300$ to grow to $\$9600$?
  [0] 30
  [1] 12
  [2] 1
  [3] 5

Time remaining: 29.9s


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


-----JSON OBJECT: {'name': 'web_search', 'arguments': {'query': 'compound interest formula to find time given final amount, initial amount, and interest rate'}}-----
---TOOL INVOCATION---
X object: {'input': 'Five thousand dollars compounded annually at an $x\\%$ interest rate takes six years to double. At the same interest rate, how many years will it take $\\$300$ to grow to $\\$9600$?', 'options': {'0': '30', '1': '12', '2': '1', '3': '5'}, 'tool_call': {'name': 'web_search', 'arguments': {'query': 'compound interest formula to find time given final amount, initial amount, and interest rate'}}}


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


========DEBUGGING ANSWER GENERATION=========
Question: Five thousand dollars compounded annually at an $x\%$ interest rate takes six years to double. At the same interest rate, how many years will it take $\$300$ to grow to $\$9600$?
Options: {'0': '30', '1': '12', '2': '1', '3': '5'}
Response: content="I used the compound interest formula to solve this problem. The correct answer is option '2': '1'.\n\nGiven that $5000 doubles in 6 years at an x% interest rate, we can first find the interest rate using the compound interest formula:\n\n\\[ A = P(1 + r)^t \\]\n\nWhere:\n- \\( A \\) is the amount of money accumulated after n years, including interest.\n- \\( P \\) is the principal amount (the initial amount of money).\n- \\( r \\) is the annual interest rate (decimal).\n- \\( t \\) is the time the money is invested for in years.\n\nFor the doubling scenario:\n\\[ 10000 = 5000(1 + r)^6 \\]\n\\[ 2 = (1 + r)^6 \\]\n\nSolving for \\( r \\):\n\\[ 1 + r = 2^{1/6} \\]\n\\[ r = 2^{1/6" addition